# Tech Challenge — Frente 4: Stack tecnológico e maturidade dos profissionais

**Pergunta central:** Qual é o stack tecnológico do profissional brasileiro de Dados e como ele está evoluindo entre diferentes senioridades e perfis?

Este notebook lê pelo AWS Glue Data Catalog as cinco tabelas da camada Gold criadas no Athena:

- dados_gold.ft_stack_profissional
- dados_gold.agg_tech_ranking_geral
- dados_gold.agg_tech_segmentos
- dados_gold.agg_tech_salario_senioridade
- dados_gold.agg_tech_evolucao_anual

O processamento analítico é realizado com PySpark. Pandas, Matplotlib e Seaborn são usados somente depois das agregações, para tabelas pequenas e visualizações.


In [ ]:
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 15


In [ ]:
# 1. Inicialização da sessão Spark e bibliotecas
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from awsglue.context import GlueContext

from io import BytesIO
import re
import unicodedata
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

print("Spark:", spark.version)


In [ ]:
# 2. Parâmetros do projeto e funções utilitárias
DB_GOLD = "dados_gold"
AMOSTRA_MINIMA = 30

SALVAR_GRAFICOS_S3 = True
BUCKET_GRAFICOS = "tech-challenge-014478672967"
PREFIXO_GRAFICOS = "artefatos/frente_4_tech"

CATEGORIAS_STACK = ["Linguagens", "Cloud", "Bancos e Plataformas", "BI"]
ORDEM_SENIORIDADE = ["Júnior", "Pleno", "Sênior"]
TECHS_CORE = ["SQL", "Python", "AWS", "Azure", "Databricks", "BigQuery", "Power BI"]

s3 = boto3.client("s3")


def nome_seguro(texto):
    """Transforma um título em nome simples e previsível para arquivo."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto.lower().strip())
    return texto.strip("_")


def concluir_grafico(fig, nome_arquivo):
    """Ajusta, salva o PNG no S3 e exibe o gráfico."""
    fig.tight_layout()

    if SALVAR_GRAFICOS_S3:
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
        buffer.seek(0)
        chave = f"{PREFIXO_GRAFICOS}/{nome_arquivo}.png"

        try:
            s3.put_object(
                Bucket=BUCKET_GRAFICOS,
                Key=chave,
                Body=buffer.getvalue(),
                ContentType="image/png"
            )
            print(f"Gráfico salvo em s3://{BUCKET_GRAFICOS}/{chave}")
        except Exception as erro:
            print("O gráfico foi exibido, mas não foi salvo no S3:", erro)

    plt.show()
    plt.close(fig)


def adicionar_rotulos_horizontais(ax, casas=1, sufixo="%"):
    """Adiciona rótulos às barras horizontais."""
    for container in ax.containers:
        labels = []
        for barra in container:
            valor = barra.get_width()
            labels.append("" if pd.isna(valor) else f"{valor:.{casas}f}{sufixo}")
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)


def adicionar_rotulos_verticais(ax, casas=1, sufixo="%"):
    """Adiciona rótulos às barras verticais."""
    for container in ax.containers:
        labels = []
        for barra in container:
            valor = barra.get_height()
            labels.append("" if pd.isna(valor) else f"{valor:.{casas}f}{sufixo}")
        ax.bar_label(container, labels=labels, padding=3, fontsize=8)


def lista_existente(lista_desejada, valores_disponiveis):
    """Mantém apenas valores realmente existentes nos dados."""
    disponiveis = set(valores_disponiveis)
    return [valor for valor in lista_desejada if valor in disponiveis]


## 1. Leitura e validação das tabelas Gold

A validação abaixo interrompe o notebook com uma mensagem clara caso alguma tabela não exista ou alguma coluna esperada esteja ausente.


In [ ]:
tabelas = {
    "fato": "ft_stack_profissional",
    "ranking": "agg_tech_ranking_geral",
    "segmentos": "agg_tech_segmentos",
    "salarios": "agg_tech_salario_senioridade",
    "evolucao": "agg_tech_evolucao_anual"
}

colunas_obrigatorias = {
    "fato": {
        "ano_pesquisa", "id_profissional", "cargo_atual", "nivel_senioridade",
        "faixa_salarial", "salario_estimado", "modelo_trabalho", "regiao",
        "flag_usa_ia", "categoria_tech", "tecnologia", "flag_adota"
    },
    "ranking": {
        "ano_pesquisa", "categoria", "tecnologia", "total_respondentes",
        "total_usuarios", "pct_adocao"
    },
    "segmentos": {
        "ano_pesquisa", "tipo_dimensao", "categoria", "tecnologia",
        "categoria_tech", "total_amostra", "total_usuarios", "pct_adocao"
    },
    "salarios": {
        "ano_pesquisa", "categoria_tech", "tecnologia", "nivel_senioridade",
        "flag_adota", "grupo_adocao", "quantidade_profissionais",
        "salario_medio", "salario_mediano"
    },
    "evolucao": {
        "ano_pesquisa", "categoria_tech", "tecnologia", "total_pesquisa_ano",
        "usuarios_tech", "pct_adocao"
    }
}


def ler_tabela_gold(apelido, tabela):
    try:
        df = (
            glueContext.create_dynamic_frame.from_catalog(
                database=DB_GOLD,
                table_name=tabela
            )
            .toDF()
        )
    except Exception as erro:
        raise RuntimeError(
            f"Não foi possível ler {DB_GOLD}.{tabela}. "
            "Confirme se a tabela foi criada no Data Catalog e se a role do Glue possui acesso."
        ) from erro

    faltantes = sorted(colunas_obrigatorias[apelido] - set(df.columns))
    if faltantes:
        raise ValueError(
            f"A tabela {DB_GOLD}.{tabela} não possui as colunas esperadas: {faltantes}"
        )

    return df.persist(StorageLevel.MEMORY_AND_DISK)


dfs = {
    apelido: ler_tabela_gold(apelido, tabela)
    for apelido, tabela in tabelas.items()
}

for apelido, df in dfs.items():
    print(
        f"{apelido:15s} | tabela: {DB_GOLD}.{tabelas[apelido]:38s} "
        f"| linhas: {df.count():7d} | colunas: {len(df.columns):2d}"
    )

df_fato = dfs["fato"]
df_ranking = dfs["ranking"]
df_segmentos = dfs["segmentos"]
df_salarios = dfs["salarios"]
df_evolucao = dfs["evolucao"]


In [ ]:
# 1.1 Cobertura e qualidade básica da tabela fato
df_cobertura = (
    df_fato
    .groupBy("ano_pesquisa")
    .agg(
        F.countDistinct("id_profissional").alias("profissionais"),
        F.count("*").alias("linhas_stack"),
        F.countDistinct(F.struct("categoria_tech", "tecnologia")).alias("tecnologias_analisadas"),
        F.sum(F.when(F.col("flag_adota").isNull(), 1).otherwise(0)).alias("flags_nulas"),
        F.sum(
            F.when(
                F.col("flag_adota").isNotNull() & (~F.col("flag_adota").isin(0, 1)),
                1
            ).otherwise(0)
        ).alias("flags_invalidas")
    )
    .orderBy("ano_pesquisa")
)

df_duplicidades = (
    df_fato
    .groupBy("ano_pesquisa", "id_profissional", "categoria_tech", "tecnologia")
    .count()
    .filter(F.col("count") > 1)
)

df_cobertura.show(truncate=False)
print("Combinações duplicadas profissional-tecnologia:", df_duplicidades.count())

ANOS_DISPONIVEIS = [
    linha["ano_pesquisa"]
    for linha in df_ranking.select("ano_pesquisa").distinct().orderBy("ano_pesquisa").collect()
]
ANO_MAIS_RECENTE = max(ANOS_DISPONIVEIS)

print("Anos disponíveis:", ANOS_DISPONIVEIS)
print("Ano mais recente:", ANO_MAIS_RECENTE)


## 2. Ranking tecnológico no ano mais recente

A taxa de adoção representa a proporção de respondentes com flag de uso igual a 1 para cada tecnologia. Os rankings são calculados separadamente para Linguagens, Cloud, Bancos/Plataformas e BI.


In [ ]:
def plotar_ranking_categoria(categoria_nome, max_itens=8):
    df_cat = (
        df_ranking
        .filter(
            (F.col("categoria") == categoria_nome) &
            (F.col("ano_pesquisa") == ANO_MAIS_RECENTE)
        )
        .orderBy(F.desc("pct_adocao"), F.desc("total_usuarios"))
        .limit(max_itens)
    )

    pdf = df_cat.toPandas().sort_values("pct_adocao", ascending=True)

    if pdf.empty:
        print(f"Sem dados para {categoria_nome} em {ANO_MAIS_RECENTE}.")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(10, max(4, len(pdf) * 0.52)))
    sns.barplot(
        data=pdf,
        x="pct_adocao",
        y="tecnologia",
        color="#2563EB",
        ax=ax
    )
    ax.set_title(f"Adoção em {categoria_nome} — {ANO_MAIS_RECENTE}")
    ax.set_xlabel("Respondentes que adotam a tecnologia (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)
    adicionar_rotulos_horizontais(ax)

    concluir_grafico(
        fig,
        f"01_ranking_{nome_seguro(categoria_nome)}_{ANO_MAIS_RECENTE}"
    )

    top = pdf.sort_values("pct_adocao", ascending=False).head(3)
    descricao = "; ".join(
        f"{linha.tecnologia}: {linha.pct_adocao:.2f}%"
        for linha in top.itertuples()
    )

    print(f"NARRATIVA — {categoria_nome.upper()}")
    print(f"Lideranças em {ANO_MAIS_RECENTE}: {descricao}.")
    print(
        "O percentual mede adoção declarada na amostra e não comprova "
        "proficiência ou profundidade de uso."
    )

    return pdf.sort_values("pct_adocao", ascending=False)


rankings_recentes = {
    categoria: plotar_ranking_categoria(categoria)
    for categoria in CATEGORIAS_STACK
}


## 3. Evolução anual do stack

Esta seção utiliza a tabela agg_tech_evolucao_anual e compara todos os anos disponíveis. Para evitar excesso de linhas, cada gráfico acompanha as tecnologias mais adotadas no ano mais recente da respectiva categoria.


In [ ]:
def plotar_evolucao_categoria(categoria_nome, max_itens=5):
    top_techs = (
        df_ranking
        .filter(
            (F.col("categoria") == categoria_nome) &
            (F.col("ano_pesquisa") == ANO_MAIS_RECENTE)
        )
        .orderBy(F.desc("pct_adocao"))
        .limit(max_itens)
        .select("tecnologia")
    )

    tecnologias = [linha["tecnologia"] for linha in top_techs.collect()]

    pdf = (
        df_evolucao
        .filter(
            (F.col("categoria_tech") == categoria_nome) &
            (F.col("tecnologia").isin(tecnologias))
        )
        .select("ano_pesquisa", "tecnologia", "pct_adocao", "total_pesquisa_ano")
        .orderBy("ano_pesquisa", "tecnologia")
        .toPandas()
    )

    if pdf.empty:
        print(f"Sem série histórica para {categoria_nome}.")
        return pd.DataFrame()

    fig, ax = plt.subplots(figsize=(11, 6))
    sns.lineplot(
        data=pdf,
        x="ano_pesquisa",
        y="pct_adocao",
        hue="tecnologia",
        marker="o",
        linewidth=2.2,
        ax=ax
    )
    ax.set_title(f"Evolução da adoção em {categoria_nome}")
    ax.set_xlabel("Ano da pesquisa")
    ax.set_ylabel("Adoção (%)")
    ax.set_xticks(sorted(pdf["ano_pesquisa"].unique()))
    ax.set_ylim(0, min(100, max(10, pdf["pct_adocao"].max() * 1.18)))
    ax.legend(title="Tecnologia", bbox_to_anchor=(1.02, 1), loc="upper left")

    concluir_grafico(fig, f"02_evolucao_{nome_seguro(categoria_nome)}")

    comparacoes = []
    for tecnologia, grupo in pdf.groupby("tecnologia"):
        grupo = grupo.sort_values("ano_pesquisa")
        if len(grupo) >= 2:
            inicial = grupo.iloc[0]
            final = grupo.iloc[-1]
            comparacoes.append({
                "tecnologia": tecnologia,
                "ano_inicial": int(inicial["ano_pesquisa"]),
                "ano_final": int(final["ano_pesquisa"]),
                "pct_inicial": float(inicial["pct_adocao"]),
                "pct_final": float(final["pct_adocao"]),
                "variacao_pp": float(final["pct_adocao"] - inicial["pct_adocao"])
            })

    pdf_variacao = pd.DataFrame(comparacoes)
    if not pdf_variacao.empty:
        maior_alta = pdf_variacao.sort_values("variacao_pp", ascending=False).iloc[0]
        maior_queda = pdf_variacao.sort_values("variacao_pp", ascending=True).iloc[0]

        print(f"NARRATIVA — EVOLUÇÃO DE {categoria_nome.upper()}")
        print(
            f"Maior aumento entre os extremos disponíveis: {maior_alta['tecnologia']} "
            f"({maior_alta['variacao_pp']:+.2f} p.p.)."
        )
        print(
            f"Menor variação ou maior queda: {maior_queda['tecnologia']} "
            f"({maior_queda['variacao_pp']:+.2f} p.p.)."
        )

    return pdf


evolucoes = {
    categoria: plotar_evolucao_categoria(categoria)
    for categoria in CATEGORIAS_STACK
}


## 4. Diferenças por senioridade e cargo

A tabela agg_tech_segmentos contém os recortes disponíveis no SQL Gold: Senioridade e Cargo. Somente grupos com pelo menos AMOSTRA_MINIMA respondentes são exibidos.


In [ ]:
# 4.1 Tecnologias estratégicas por senioridade
tecnologias_segmento = [
    linha["tecnologia"]
    for linha in df_segmentos.select("tecnologia").distinct().collect()
]
techs_core_segmento = lista_existente(TECHS_CORE, tecnologias_segmento)

df_sen = (
    df_segmentos
    .filter(
        (F.col("tipo_dimensao") == "Senioridade") &
        (F.col("ano_pesquisa") == ANO_MAIS_RECENTE) &
        (F.col("tecnologia").isin(techs_core_segmento)) &
        (F.col("total_amostra") >= AMOSTRA_MINIMA)
    )
)

pdf_sen = df_sen.toPandas()

if pdf_sen.empty:
    print("Não há grupos de senioridade com a amostra mínima definida.")
else:
    ordem_senioridade_existente = lista_existente(
        ORDEM_SENIORIDADE,
        pdf_sen["categoria"].dropna().unique()
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(
        data=pdf_sen,
        x="tecnologia",
        y="pct_adocao",
        hue="categoria",
        hue_order=ordem_senioridade_existente,
        palette=["#93C5FD", "#3B82F6", "#1E3A8A"][:len(ordem_senioridade_existente)],
        ax=ax
    )
    ax.set_title(f"Adoção de tecnologias por senioridade — {ANO_MAIS_RECENTE}")
    ax.set_xlabel("Tecnologia")
    ax.set_ylabel("Adoção (%)")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=25)
    ax.legend(title="Senioridade")
    adicionar_rotulos_verticais(ax, casas=1)

    concluir_grafico(fig, f"03_tecnologia_senioridade_{ANO_MAIS_RECENTE}")

    pivot_sen = pdf_sen.pivot_table(
        index="tecnologia",
        columns="categoria",
        values="pct_adocao",
        aggfunc="first"
    )

    if {"Júnior", "Sênior"}.issubset(pivot_sen.columns):
        pivot_sen["diferenca_senior_menos_junior_pp"] = (
            pivot_sen["Sênior"] - pivot_sen["Júnior"]
        )
        maior_gap = (
            pivot_sen
            .dropna(subset=["diferenca_senior_menos_junior_pp"])
            .sort_values("diferenca_senior_menos_junior_pp", ascending=False)
        )

        if not maior_gap.empty:
            tecnologia_gap = maior_gap.index[0]
            valor_gap = maior_gap.iloc[0]["diferenca_senior_menos_junior_pp"]
            print("NARRATIVA — SENIORIDADE")
            print(
                f"A maior diferença entre Sênior e Júnior ocorreu em "
                f"{tecnologia_gap}: {valor_gap:+.2f} pontos percentuais."
            )


In [ ]:
# 4.2 Mapa de adoção por cargo
df_cargo = (
    df_segmentos
    .filter(
        (F.col("tipo_dimensao") == "Cargo") &
        (F.col("ano_pesquisa") == ANO_MAIS_RECENTE) &
        (F.col("tecnologia").isin(techs_core_segmento)) &
        (F.col("total_amostra") >= AMOSTRA_MINIMA)
    )
    .select("categoria", "tecnologia", "total_amostra", "pct_adocao")
)

pdf_cargo = df_cargo.toPandas()

if pdf_cargo.empty:
    print("Não há cargos com a amostra mínima definida.")
else:
    top_cargos = (
        pdf_cargo
        .groupby("categoria")["total_amostra"]
        .max()
        .nlargest(10)
        .index
    )

    matriz_cargo = (
        pdf_cargo[pdf_cargo["categoria"].isin(top_cargos)]
        .pivot_table(
            index="categoria",
            columns="tecnologia",
            values="pct_adocao",
            aggfunc="first"
        )
    )

    fig, ax = plt.subplots(figsize=(12, max(5, len(matriz_cargo) * 0.52)))
    sns.heatmap(
        matriz_cargo,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        vmin=0,
        vmax=100,
        cbar_kws={"label": "Adoção (%)"},
        ax=ax
    )
    ax.set_title(f"Adoção tecnológica por cargo — {ANO_MAIS_RECENTE}")
    ax.set_xlabel("Tecnologia")
    ax.set_ylabel("Cargo")

    concluir_grafico(fig, f"04_tecnologia_cargo_{ANO_MAIS_RECENTE}")

    maior_celula = matriz_cargo.stack().sort_values(ascending=False)
    if not maior_celula.empty:
        cargo_lider, tecnologia_lider = maior_celula.index[0]
        print("NARRATIVA — CARGO")
        print(
            f"Entre os cargos e tecnologias exibidos, a maior adoção foi "
            f"{tecnologia_lider} em {cargo_lider}: {maior_celula.iloc[0]:.2f}%."
        )


## 5. Amplitude do stack como indicador de maturidade

A amplitude corresponde à quantidade de tecnologias marcadas por profissional dentro do recorte da tabela fato. Ela não representa domínio técnico e está limitada às tecnologias selecionadas no SQL Gold.


In [ ]:
# Uma linha por profissional, com a quantidade de tecnologias adotadas
df_amplitude_profissional = (
    df_fato
    .groupBy(
        "ano_pesquisa",
        "id_profissional",
        "nivel_senioridade",
        "flag_usa_ia"
    )
    .agg(
        F.sum(F.coalesce(F.col("flag_adota"), F.lit(0))).alias(
            "qtd_tecnologias_adotadas"
        )
    )
    .withColumn(
        "flag_usa_ia_norm",
        F.when(
            F.lower(F.trim(F.col("flag_usa_ia").cast("string"))).isin("1", "true"),
            F.lit(1)
        )
        .when(
            F.lower(F.trim(F.col("flag_usa_ia").cast("string"))).isin("0", "false"),
            F.lit(0)
        )
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

df_amplitude_senioridade = (
    df_amplitude_profissional
    .filter(F.col("nivel_senioridade").isin(ORDEM_SENIORIDADE))
    .groupBy("ano_pesquisa", "nivel_senioridade")
    .agg(
        F.count("*").alias("profissionais"),
        F.round(F.avg("qtd_tecnologias_adotadas"), 2).alias("media_tecnologias"),
        F.expr(
            "percentile_approx(qtd_tecnologias_adotadas, 0.5)"
        ).alias("mediana_tecnologias")
    )
    .filter(F.col("profissionais") >= AMOSTRA_MINIMA)
    .orderBy("ano_pesquisa", "nivel_senioridade")
)

pdf_amplitude_sen = df_amplitude_senioridade.toPandas()
df_amplitude_senioridade.show(truncate=False)

if not pdf_amplitude_sen.empty:
    ordem_existente = lista_existente(
        ORDEM_SENIORIDADE,
        pdf_amplitude_sen["nivel_senioridade"].unique()
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(
        data=pdf_amplitude_sen,
        x="ano_pesquisa",
        y="media_tecnologias",
        hue="nivel_senioridade",
        hue_order=ordem_existente,
        marker="o",
        linewidth=2.4,
        palette=["#93C5FD", "#3B82F6", "#1E3A8A"][:len(ordem_existente)],
        ax=ax
    )
    ax.set_title("Amplitude média do stack por senioridade")
    ax.set_xlabel("Ano da pesquisa")
    ax.set_ylabel("Média de tecnologias adotadas")
    ax.set_xticks(sorted(pdf_amplitude_sen["ano_pesquisa"].unique()))
    ax.legend(title="Senioridade")

    concluir_grafico(fig, "05_amplitude_stack_senioridade")

    recorte_atual = pdf_amplitude_sen[
        pdf_amplitude_sen["ano_pesquisa"] == ANO_MAIS_RECENTE
    ]

    if not recorte_atual.empty:
        maior = recorte_atual.sort_values("media_tecnologias", ascending=False).iloc[0]
        menor = recorte_atual.sort_values("media_tecnologias", ascending=True).iloc[0]
        print("NARRATIVA — AMPLITUDE DO STACK")
        print(
            f"Em {ANO_MAIS_RECENTE}, a maior amplitude média ocorreu em "
            f"{maior['nivel_senioridade']} ({maior['media_tecnologias']:.2f}) "
            f"e a menor em {menor['nivel_senioridade']} "
            f"({menor['media_tecnologias']:.2f})."
        )


In [ ]:
# 5.1 Amplitude do stack e uso de IA
df_amplitude_ia = (
    df_amplitude_profissional
    .filter(F.col("flag_usa_ia_norm").isin(0, 1))
    .groupBy("ano_pesquisa", "flag_usa_ia_norm")
    .agg(
        F.count("*").alias("profissionais"),
        F.round(F.avg("qtd_tecnologias_adotadas"), 2).alias("media_tecnologias")
    )
    .filter(F.col("profissionais") >= AMOSTRA_MINIMA)
    .withColumn(
        "grupo_ia",
        F.when(F.col("flag_usa_ia_norm") == 1, "Usa IA").otherwise("Não usa IA")
    )
    .orderBy("ano_pesquisa", "flag_usa_ia_norm")
)

pdf_amplitude_ia = df_amplitude_ia.toPandas()
df_amplitude_ia.show(truncate=False)

if not pdf_amplitude_ia.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(
        data=pdf_amplitude_ia,
        x="ano_pesquisa",
        y="media_tecnologias",
        hue="grupo_ia",
        marker="o",
        linewidth=2.4,
        palette={"Não usa IA": "#94A3B8", "Usa IA": "#2563EB"},
        ax=ax
    )
    ax.set_title("Amplitude média do stack e uso de IA")
    ax.set_xlabel("Ano da pesquisa")
    ax.set_ylabel("Média de tecnologias adotadas")
    ax.set_xticks(sorted(pdf_amplitude_ia["ano_pesquisa"].unique()))
    ax.legend(title="Grupo")

    concluir_grafico(fig, "06_amplitude_stack_uso_ia")

    print(
        "Interpretação: diferenças entre os grupos indicam associação. "
        "Não demonstram que o uso de IA cause maior amplitude tecnológica."
    )


## 6. Associação entre tecnologia, salário e senioridade

A comparação é feita dentro da mesma senioridade e exige pelo menos AMOSTRA_MINIMA profissionais nos grupos que adotam e não adotam cada tecnologia. O notebook escolhe automaticamente o ano mais recente que possui pares comparáveis.


In [ ]:
tecnologias_salario_disponiveis = [
    linha["tecnologia"]
    for linha in df_salarios.select("tecnologia").distinct().collect()
]
techs_salario = lista_existente(TECHS_CORE, tecnologias_salario_disponiveis)

anos_salario = [
    linha["ano_pesquisa"]
    for linha in (
        df_salarios
        .select("ano_pesquisa")
        .distinct()
        .orderBy(F.desc("ano_pesquisa"))
        .collect()
    )
]

ano_salario = None
pares_validos_ano = None

for ano in anos_salario:
    candidatos = (
        df_salarios
        .filter(
            (F.col("ano_pesquisa") == ano) &
            (F.col("tecnologia").isin(techs_salario)) &
            (F.col("nivel_senioridade").isin(ORDEM_SENIORIDADE)) &
            (F.col("quantidade_profissionais") >= AMOSTRA_MINIMA) &
            (F.col("salario_mediano").isNotNull())
        )
    )

    pares = (
        candidatos
        .groupBy("tecnologia", "nivel_senioridade")
        .agg(F.countDistinct("flag_adota").alias("grupos_comparaveis"))
        .filter(F.col("grupos_comparaveis") == 2)
    )

    if pares.limit(1).count() > 0:
        ano_salario = ano
        pares_validos_ano = pares.select("tecnologia", "nivel_senioridade")
        break

if ano_salario is None:
    print(
        "Nenhum ano possui pares comparáveis com a amostra mínima. "
        "Considere reduzir AMOSTRA_MINIMA com justificativa metodológica."
    )
    pdf_salario = pd.DataFrame()
    pdf_diferencas_salariais = pd.DataFrame()
else:
    df_salario_comparavel = (
        df_salarios
        .filter(
            (F.col("ano_pesquisa") == ano_salario) &
            (F.col("tecnologia").isin(techs_salario)) &
            (F.col("nivel_senioridade").isin(ORDEM_SENIORIDADE)) &
            (F.col("quantidade_profissionais") >= AMOSTRA_MINIMA) &
            (F.col("salario_mediano").isNotNull())
        )
        .join(
            F.broadcast(pares_validos_ano),
            on=["tecnologia", "nivel_senioridade"],
            how="inner"
        )
        .select(
            "tecnologia",
            "nivel_senioridade",
            "flag_adota",
            "grupo_adocao",
            "quantidade_profissionais",
            "salario_mediano"
        )
    )

    pdf_salario = df_salario_comparavel.toPandas()
    print("Ano selecionado para comparação salarial:", ano_salario)

    niveis_presentes = lista_existente(
        ORDEM_SENIORIDADE,
        pdf_salario["nivel_senioridade"].unique()
    )

    fig, axes = plt.subplots(
        1,
        len(niveis_presentes),
        figsize=(7 * len(niveis_presentes), 6),
        sharey=False
    )

    if len(niveis_presentes) == 1:
        axes = [axes]

    for ax, nivel in zip(axes, niveis_presentes):
        dados_nivel = pdf_salario[
            pdf_salario["nivel_senioridade"] == nivel
        ].copy()

        sns.barplot(
            data=dados_nivel,
            x="tecnologia",
            y="salario_mediano",
            hue="grupo_adocao",
            hue_order=["Nao Adota", "Adota Tecnologia"],
            palette={"Nao Adota": "#94A3B8", "Adota Tecnologia": "#16A34A"},
            ax=ax
        )
        ax.set_title(nivel)
        ax.set_xlabel("")
        ax.set_ylabel("Salário mediano (R$)")
        ax.tick_params(axis="x", rotation=35)
        ax.yaxis.set_major_formatter(
            mtick.FuncFormatter(lambda valor, pos: f"R$ {valor:,.0f}")
        )
        ax.legend(title="Grupo", fontsize=8)

    fig.suptitle(
        f"Salário mediano por adoção tecnológica e senioridade — {ano_salario}",
        y=1.03,
        fontsize=15
    )

    concluir_grafico(fig, f"07_salario_tecnologia_senioridade_{ano_salario}")

    pivot_salario = pdf_salario.pivot_table(
        index=["tecnologia", "nivel_senioridade"],
        columns="grupo_adocao",
        values="salario_mediano",
        aggfunc="first"
    )

    if {"Nao Adota", "Adota Tecnologia"}.issubset(pivot_salario.columns):
        pivot_salario["diferenca_percentual"] = (
            100.0 *
            (
                pivot_salario["Adota Tecnologia"] -
                pivot_salario["Nao Adota"]
            ) /
            pivot_salario["Nao Adota"]
        )

        pdf_diferencas_salariais = (
            pivot_salario
            .reset_index()
            .sort_values("diferenca_percentual", ascending=False)
        )

        print("Comparação salarial dentro da mesma senioridade:")
        print(
            pdf_diferencas_salariais[
                [
                    "tecnologia",
                    "nivel_senioridade",
                    "Nao Adota",
                    "Adota Tecnologia",
                    "diferenca_percentual"
                ]
            ].round(2).to_string(index=False)
        )
        print(
            "Interpretação: as diferenças são associações observadas. "
            "Não concluem que adotar a tecnologia causou o salário."
        )
    else:
        pdf_diferencas_salariais = pd.DataFrame()


## 7. Síntese estratégica orientada pelos resultados

A síntese abaixo é calculada pelo notebook. Assim, o texto final acompanha os dados efetivamente catalogados, sem assumir previamente quais tecnologias lideram ou apresentam maior crescimento.


In [ ]:
# Líder de cada categoria no ano mais recente
janela_ranking = Window.partitionBy("categoria").orderBy(
    F.desc("pct_adocao"),
    F.desc("total_usuarios"),
    F.asc("tecnologia")
)

lideres = (
    df_ranking
    .filter(F.col("ano_pesquisa") == ANO_MAIS_RECENTE)
    .withColumn("posicao", F.row_number().over(janela_ranking))
    .filter(F.col("posicao") == 1)
    .select("categoria", "tecnologia", "pct_adocao", "total_usuarios")
    .orderBy("categoria")
    .collect()
)

print("=" * 88)
print("SÍNTESE DA FRENTE 4 — STACK TECNOLÓGICO E MATURIDADE")
print("=" * 88)
print(f"Período analisado: {min(ANOS_DISPONIVEIS)} a {max(ANOS_DISPONIVEIS)}.")
print(f"Ano mais recente dos rankings: {ANO_MAIS_RECENTE}.")
print()

print("1. STACK LÍDER POR CATEGORIA")
for linha in lideres:
    print(
        f"- {linha['categoria']}: {linha['tecnologia']} "
        f"({linha['pct_adocao']:.2f}% de adoção; "
        f"{linha['total_usuarios']} usuários)."
    )

print()
print("2. MATURIDADE E AMPLITUDE DO STACK")
recorte_amplitude = pdf_amplitude_sen[
    pdf_amplitude_sen["ano_pesquisa"] == ANO_MAIS_RECENTE
].sort_values("media_tecnologias", ascending=False)

if recorte_amplitude.empty:
    print("- Não houve recorte com a amostra mínima no ano mais recente.")
else:
    for linha in recorte_amplitude.itertuples():
        print(
            f"- {linha.nivel_senioridade}: média de "
            f"{linha.media_tecnologias:.2f} tecnologias no recorte analisado."
        )

print()
print("3. REMUNERAÇÃO")
if ano_salario is None or pdf_diferencas_salariais.empty:
    print("- Não houve pares comparáveis suficientes para estimar diferenças salariais.")
else:
    maior_assoc = pdf_diferencas_salariais.iloc[0]
    menor_assoc = pdf_diferencas_salariais.iloc[-1]
    print(f"- Ano comparável selecionado: {ano_salario}.")
    print(
        f"- Maior diferença positiva observada: "
        f"{maior_assoc['tecnologia']} / {maior_assoc['nivel_senioridade']} "
        f"({maior_assoc['diferenca_percentual']:+.2f}%)."
    )
    print(
        f"- Menor diferença ou maior diferença negativa: "
        f"{menor_assoc['tecnologia']} / {menor_assoc['nivel_senioridade']} "
        f"({menor_assoc['diferenca_percentual']:+.2f}%)."
    )
    print("- Essas diferenças representam associação, não efeito causal.")

print()
print("4. OPORTUNIDADES E RECOMENDAÇÕES")
print(
    "- Priorizar capacitação nas tecnologias líderes e nas que apresentam "
    "crescimento consistente em mais de um ano."
)
print(
    "- Direcionar trilhas por senioridade com base nos maiores gaps de adoção, "
    "sempre respeitando a amostra mínima."
)
print(
    "- Usar a amplitude do stack como indicador de exposição tecnológica, "
    "não como prova de proficiência."
)
print(
    "- Tratar resultados salariais como sinais para investigação, controlando "
    "cargo, experiência, região e porte da empresa em análises futuras."
)


## 8. Limitações metodológicas

- Adoção declarada não equivale a proficiência ou domínio da tecnologia.
- A tabela fato contempla apenas as tecnologias selecionadas no SQL Gold; portanto, a amplitude mede esse recorte, não todo o ecossistema tecnológico.
- O salário estimado deriva das variáveis disponíveis na pesquisa e pode não representar a remuneração exata.
- Comparações salariais são associações observacionais e não permitem afirmar causalidade.
- Categorias com menos de AMOSTRA_MINIMA respondentes são omitidas para reduzir conclusões instáveis.
- Alterações de questionário, cobertura e composição da amostra entre os anos podem influenciar a evolução observada.
